In [3]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [5]:
api_wrapper = WikipediaAPIWrapper(top_k_results=10, doc_content_chars_max=2000)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [7]:
tool.name

'wikipedia'

In [10]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

vectordb = FAISS.from_documents(documents, OllamaEmbeddings(model="qwen3-embedding:8b"))
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000016AB0377650>, search_kwargs={})

In [11]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "search for unformation about LangSmith. For any question about LangthSmith, you must use this tool!"
    )

In [13]:
retriever_tool.name

'langsmith_search'

In [16]:
#3 arxiv tool

from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=10, doc_content_chars_max=2000)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [17]:
tools=[tool, retriever_tool, arxiv]

In [18]:
tools


[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\AI Engineering\\Langchain chat\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=10, lang='en', load_all_available_meta=False, doc_content_chars_max=2000)),
 StructuredTool(name='langsmith_search', description='search for unformation about LangSmith. For any question about LangthSmith, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000016B4B20A980>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000016B4B20A8E0>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=10, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=2000))]

In [19]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3.8")

In [26]:
from langchain_classic import hub

prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

ValueError: Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted serialized LangChain objects. If you trust this prompt, set `dangerously_pull_public_prompt=True` to acknowledge the risk.

In [27]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),

    MessagesPlaceholder(
        variable_name="chat_history",
        optional=True
    ),

    ("human", "{input}"),

    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])

print(prompt)

input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]

In [ ]:
### agents

from langchain.agents import create_agent

agent = create_agent(
    llm, 
    tools,
    prompt="You are a helpful assistant. You should only use tools to answer.")

In [ ]:
### agent executor
# old deprecated for current agents

from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import agent.

runnable_agent = RunnableAgent(
    runnable_agent=agent,
    stream_runnable=False
)

agent_executor = AgentExecutor(agent=runnable_agent, tools=tools, verbose=True)
agent_executor

ImportError: cannot import name 'RunnableAgent' from 'langchain_classic.agents' (d:\AI Engineering\Langchain chat\venv\Lib\site-packages\langchain_classic\agents\__init__.py)

In [49]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the langsmith"}]})

response["messages"][-1].content

"# LangSmith\n\n**LangSmith** is a platform developed by LangChain for building, debugging, testing, monitoring, and evaluating LLM (Large Language Model) applications. It serves as a comprehensive observability and lifecycle management tool for AI/LLM-powered applications.\n\n## Key Features\n\n- **Tracing & Observability**: Provides detailed, end-to-end tracing of LLM application workflows, allowing developers to see every step of a prompt, chain, or agent execution.\n- **Debugging**: Helps developers identify and debug issues in complex LLM pipelines by visualizing the flow of data through chains, agents, and tool calls.\n- **Testing & Evaluation**: Enables systematic evaluation of LLM outputs using datasets, metrics, and regression testing.\n- **Monitoring**: Tracks application performance, latency, costs, and error rates in production.\n- **Dataset Management**: Allows you to create, version, and manage datasets for fine-tuning, evaluation, and testing.\n- **Prompt Management**: C

In [46]:
agent.invoke(
    {"messages": [{"role": "user", "content": "what is machine learning"}]})

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [47]:
agent.invoke(
    {"messages": [{"role": "user", "content": "What's the paper 1605.08386 about?"}]})

AttributeError: 'Search' object has no attribute 'results'

Old tools wont do.